# Anti-Money Laundering (AML) System: RAW & SAR Agents with A2A Communication

This notebook is inspired by the multi-agent AML architecture and implemented using this project’s local modules.

## What this notebook covers
- RAW Agent (rule-based watchdog)
- SAR Agent (ML + graph signals)
- A2A-style orchestration between agents
- SAR-style report generation
- Batch metrics summary

## 1) Environment & project path setup
This notebook runs with your local project files only (no runtime dataset downloads).

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / 'app').exists() is False and (PROJECT_ROOT.parent / 'app').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('App exists:', (PROJECT_ROOT / 'app').exists())
print('Data exists:', (PROJECT_ROOT / 'data').exists())

## 2) Imports
Imports are intentionally local-first and aligned with this repository’s modules.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

from app.services.agentic_aml import AMLOrchestratorA2A, SARReportGenerator, batch_metrics

print('Imports successful ✅')

## 3) Load local dataset
It prefers `data/training_transactions.csv`; if not found, it uses the template file for demo.

In [ ]:
primary_path = PROJECT_ROOT / 'data' / 'training_transactions.csv'
fallback_path = PROJECT_ROOT / 'data' / 'training_transactions.template.csv'

data_path = primary_path if primary_path.exists() else fallback_path
if not data_path.exists():
    raise FileNotFoundError('No local dataset found in data/. Add training_transactions.csv or template.')

df = pd.read_csv(data_path)
print('Using dataset:', data_path)
print('Rows:', len(df), 'Columns:', list(df.columns))
df.head()

## 4) Normalize for agentic transaction flow
Builds transaction-like records expected by the orchestrator (`amount`, `src_account`, `dst_account`, etc.).

In [ ]:
tx_df = df.copy()

if 'transaction_amount' in tx_df.columns and 'amount' not in tx_df.columns:
    tx_df['amount'] = tx_df['transaction_amount']

if 'tx_count_last_hour' not in tx_df.columns:
    tx_df['tx_count_last_hour'] = 1

if 'src_account' not in tx_df.columns:
    tx_df['src_account'] = [f'ACC_{i:06d}' for i in range(len(tx_df))]

if 'dst_account' not in tx_df.columns:
    tx_df['dst_account'] = [f'ACC_{i+1000:06d}' for i in range(len(tx_df))]

if 'transaction_note' not in tx_df.columns:
    tx_df['transaction_note'] = np.where(
        tx_df.get('label', pd.Series([0]*len(tx_df))).astype(int) == 1,
        'urgent mule cashout pattern',
        'normal customer transfer'
    )

if 'transaction_id' not in tx_df.columns:
    tx_df['transaction_id'] = [f'TXN_{i:08d}' for i in range(len(tx_df))]

tx_df[['transaction_id', 'amount', 'tx_count_last_hour', 'src_account', 'dst_account', 'transaction_note']].head()

## 5) Initialize RAW + SAR + A2A orchestrator

In [ ]:
orchestrator = AMLOrchestratorA2A()
print('Orchestrator ready ✅')

## 6) Process a sample transaction (single inference)

In [ ]:
sample_txn = tx_df.iloc[0].to_dict()
single_result = orchestrator.process(sample_txn)

single_view = {
    'transaction_id': single_result.transaction_id,
    'final_decision': single_result.final_decision,
    'combined_risk_score': single_result.combined_risk_score,
    'raw_action': single_result.raw_decision.action,
    'raw_risk': single_result.raw_decision.risk_score,
    'sar_ensemble': single_result.sar_decision.ensemble_score,
    'reason': single_result.reason
}
single_view

## 7) Generate SAR-style report

In [ ]:
report = SARReportGenerator.build_report(single_result, sample_txn)
print('Report ID:', report['report_id'])
print('Final action:', report['final_decision']['action'])
print('Combined score:', report['final_decision']['combined_risk_score'])

report_path = PROJECT_ROOT / 'artifacts' / 'notebook_sample_report.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved:', report_path)

## 8) Batch run + AML metrics
Treat `ESCALATE`/`BLOCK` as positive alerts.

In [ ]:
batch_size = min(200, len(tx_df))
batch_df = tx_df.head(batch_size)
results = [orchestrator.process(row.to_dict()) for _, row in batch_df.iterrows()]

summary = batch_metrics(results)
print('Summary:', summary)

if 'label' in batch_df.columns:
    y_true = batch_df['label'].astype(int).tolist()
    y_pred = [1 if r.final_decision in {'ESCALATE', 'BLOCK'} else 0 for r in results]

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred).tolist()

    print('Precision:', round(float(precision), 4))
    print('Recall   :', round(float(recall), 4))
    print('F1-score :', round(float(f1), 4))
    print('Confusion matrix [ [tn, fp], [fn, tp] ]:', cm)
else:
    print('No label column found; skipped supervised metrics.')

## 9) Export notebook batch artifact

In [ ]:
artifact = {
    'summary': summary,
    'sample_transaction_id': single_result.transaction_id,
    'sample_decision': single_result.final_decision,
    'sample_combined_risk': single_result.combined_risk_score
}

artifact_path = PROJECT_ROOT / 'artifacts' / 'notebook_batch_summary.json'
artifact_path.write_text(json.dumps(artifact, indent=2), encoding='utf-8')
print('Saved:', artifact_path)
artifact

## 10) Next Steps
- Swap template CSV with larger local real dataset in `data/training_transactions.csv`.
- Retrain model using `scripts/train_ml.py` (XGBoost + SHAP).
- Compare notebook metrics before/after feedback-driven weight tuning via API endpoints.